In [1]:
from collections import OrderedDict

import numpy as np
from qiskit_metal import designs, Dict, MetalGUI
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.qubits.circle_transmon import CircTransmon

# custom components (live in the qcg-quantum-design env's qiskit_metal.qlibrary)
from qiskit_metal.qlibrary.qubits.squid_loop_jj import SquidLoopJJ
from qiskit_metal.qlibrary.qubits.squid_array import SquidArray
from qiskit_metal.qlibrary.sample_shapes.rect_cutout import RectCutout

from build_design import (
    L_TRANSMON_NH, L_PROBE_NH, CJ_TRANSMON_FF, CJ_PROBE_FF,
    QUBIT_COMPONENT_NAME, RES_LENGTH_MM,
)

design = designs.DesignPlanar({}, overwrite_enabled=True)
gui = MetalGUI(design)
design.variables['cpw_width'] = '20um'
design.variables['cpw_gap'] = '12.25um'

## SQUID components

Three custom components (defined in `qiskit_metal/qlibrary/` of the `qcg-quantum-design` env):

| component | module | what it does |
|-----------|--------|--------------|
| `SquidLoopJJ` | `qubits.squid_loop_jj` | `SQUID_LOOP` + two real Josephson junctions carved into the `JJ_gap`s + a ground-plane pocket so those gaps are not shorted |
| `SquidArray` | `qubits.squid_array` | `n_squids` SQUID loops tiled and wired **in series** (a tunable superinductor), pins `a`/`b` at the ends |
| `RectCutout` | `sample_shapes.rect_cutout` | one `subtract=True` rectangle — a general ground-plane cutout |

### How gaps and junctions work in qiskit-metal

The ground plane is implicit: each renderer lays a chip-wide metal sheet, unions in every
`subtract=False` (positive) row, then subtracts every `subtract=True` row. So:

* **carving a gap** = emit a shapely polygon as a `subtract=True` `poly` row —
  `self.add_qgeometry('poly', {'g': shape}, subtract=True, chip=p.chip)`. Build `shape` with
  `draw.rectangle`, `draw.Point(...).buffer(r)`, `draw.LineString([...]).buffer(w/2, cap_style=2)`,
  `draw.Polygon(...)`, combined via `draw.union` / `draw.subtract` / `.buffer`.
* **a junction** = a short `LineString` bridging a real metal break, emitted to the `junction`
  table with `width`, `hfss_inductance` (`L_j`) and `hfss_capacitance` (`C_j`).

**A bare `JJ_gap` still shorts the arms** unless the ground plane is also cleared there —
`SQUID_LOOP` never carves a pocket, so `SquidLoopJJ`/`SquidArray` add one (the same trick as
`CircTransmonSQUID._add_squid_background`).

**Carving around one SQUID does not eat its neighbours' metal.** `subtract=True` rows are
removed from the ground plane *only*; the renderer re-adds every positive polygon on top
untouched (see `_positive_mask` in the GDS renderer). Overlapping pockets from adjacent SQUIDs
just merge into one trench — which is what a series chain wants anyway.


In [2]:
# --- SQUID array: N loops in series (tunable superinductor) ----------------

def build_squid_array(name, n_squids, pos_x_um, pos_y_um, orientation,
                      squid_gap_um, arm_width_um, josephson_gap_um,
                      segment_length_um, arm_semi_length_um, line_width_um,
                      L_j_nH, C_j_fF, jj_width_um, ground_gap_um=3.0):
    return SquidArray(design, name, options={
        'pos_x': f'{pos_x_um}um',
        'pos_y': f'{pos_y_um}um',
        'orientation': f'{orientation}',
        'n_squids': f'{n_squids}',
        'plate1_width': f'{arm_width_um}um',
        'squid_gap': f'{squid_gap_um}um',
        'segment_a_length': f'{arm_semi_length_um}um',
        'segment_a_width': f'{arm_width_um}um',
        'JJ_gap': f'{josephson_gap_um}um',
        'segment_b_length': f'{arm_semi_length_um}um',
        'segment_b_width': f'{arm_width_um}um',
        'segment_c_width': f'{arm_width_um}um',
        'segment_d_length': f'{segment_length_um}um',
        'segment_d_width': f'{line_width_um}um',
        'jj_width': f'{jj_width_um}um',
        'L_j': f'{L_j_nH}nH',
        'C_j': f'{C_j_fF}fF',
        'ground_gap': f'{ground_gap_um}um',
    })


try:
    array_1.delete()
except NameError:
    pass

array_1 = build_squid_array(
    name="array_1", n_squids=23, pos_x_um=-1290, pos_y_um=0, orientation=0.0,
    squid_gap_um=25, arm_width_um=4, josephson_gap_um=3,
    segment_length_um=6, arm_semi_length_um=15, line_width_um=9,
    L_j_nH=L_PROBE_NH, C_j_fF=CJ_PROBE_FF, jj_width_um=2.0,
)

gui.rebuild()
gui.autoscale()

# 8 loops -> 16 junction rows, 9 galvanic nets; pins array_1:a (left) / array_1:b (right)

In [3]:
try:
    launch_point1.delete()
    launch_point2.delete()
except NameError:
    pass

# Readout-line positions derived from array_1 so every side keeps the SAME gap
# to the squids. array_1 here: pos (-1290, 0) um, 23 loops, squid_gap 25,
# arm_width 4, seg_a 15, JJ_gap 3, seg_d 6  ->
#   loop centre -> top array metal : 0.5*squid_gap + arm_width           = 16.5 um
#   loop pitch  : plate1 + 2*seg_a + JJ_gap + seg_a_w + seg_d            = 47 um
#   array metal spans x in [-1292, -211] um
# ROUTE_CLEARANCE_UM = gap from the wire centreline to the nearest array metal,
# applied on the top, bottom, left-elbow AND termination sides alike.
ROUTE_CLEARANCE_UM = 35.0

pass_y_um  =   16.5 + ROUTE_CLEARANCE_UM        # top / bottom pass
elbow_x_um = -1292.0 - ROUTE_CLEARANCE_UM       # left U-turn
term_x_um  =  -211.0 + ROUTE_CLEARANCE_UM       # both terminations (right end)

launch_point1 = OpenToGround(design, 'launch_point1', options=dict(
    pos_x=f'{term_x_um}um', pos_y=f'{pass_y_um}um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

launch_point2 = ShortToGround(design, 'launch_point2', options=dict(
    pos_x=f'{term_x_um}um', pos_y=f'{-pass_y_um}um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

gui.rebuild()
gui.autoscale()

In [4]:
try:
    readout_1.delete()
except NameError:
    pass

# RouteAnchors: start, end and two waypoints -> Manhattan segments + fillets,
# NO prescribed length (that is what threw "length doesn't fit" with
# RouteMeander). Waypoints are np.array([x, y]) in mm.
anchors = OrderedDict()
anchors[0] = np.array([elbow_x_um,  pass_y_um]) * 1e-3   # mm; left end, top pass
anchors[1] = np.array([elbow_x_um, -pass_y_um]) * 1e-3   # U-turn onto the bottom pass

readout_1 = RouteAnchors(design, 'readout_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='launch_point1', pin='open'),
        end_pin=dict(component='launch_point2', pin='short'),
    ),
    fillet='20um',                       # keep < ROUTE_CLEARANCE_UM
    trace_width='cpw_width',
    trace_gap='cpw_gap',
    lead=dict(start_straight='30um', end_straight='30um'),
    anchors=anchors,
))

gui.rebuild()
gui.autoscale()
print(f'readout_1 length = {readout_1.length * 1e3:.0f} um')

readout_1 length = 2388 um
